In [1]:
from utils.Linkedin import Linkedin
from utils.LinkedinJobApply import LinkedinJobApply
import pandas as pd
from utils.data_handle import dataframe_to_sqlite, linkedin_extract_job_details, linkedin_check_if_easy_apply_avilable, read_from_sqlite
from utils.config import *
from tqdm import tqdm
tqdm.pandas()

# Training the form QA model

## Preprocesing the pre existing correct data 
- need check and do manual Ground Truth 

In [2]:
# LinkedinJobApply.run()

In [3]:
df= read_from_sqlite(LINKEDIN_DB_FILE, LINKEDIN_FORM_QA_TABLE)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4722 entries, 0 to 4721
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   question              4722 non-null   object
 1   pre_ans               4722 non-null   object
 2   predicted_ans         251 non-null    object
 3   input_type            4722 non-null   object
 4   available_options     4722 non-null   object
 5   applied_successfully  4722 non-null   object
 6   inserted_at           4722 non-null   object
dtypes: object(7)
memory usage: 258.4+ KB


In [4]:
df.iloc[8]["available_options"]

'["Select an option", "Yes", "No"]'

In [13]:
df[df["input_type"]== 'radio']

,question,pre_ans,predicted_ans,input_type,available_options,applied_successfully,inserted_at
5,Are you comfortable working in an onsite setting?,Yes,None,radio,"[""Yes"", ""No""]",N,2025-01-23 00:36:07
7,Are you comfortable commuting to this job's lo...,,None,radio,"[""Yes"", ""No""]",N,2025-01-23 00:36:07
16,Are you comfortable working in an onsite setting?,Yes,None,radio,"[""Yes"", ""No""]",N,2025-01-23 00:39:02
18,Are you comfortable commuting to this job's lo...,,None,radio,"[""Yes"", ""No""]",N,2025-01-23 00:39:02
70,"Are you willing to undergo a background check,...",Yes,None,radio,"[""Yes"", ""No""]",N,2025-01-23 00:51:40
...,...,...,...,...,...,...,...
4499,Have you completed the following level of educ...,Yes,None,radio,"[""Yes"", ""No""]",N,2025-01-23 06:37:42
4519,Are you comfortable working in an onsite setting?,Yes,None,radio,"[""Yes"", ""No""]",N,2025-01-23 06:39:21
4520,Are you comfortable commuting to this job's lo...,,Yes,radio,"[""Yes"", ""No""]",N,2025-01-23 06:39:21
4665,Have you completed the following level of educ...,No,None,radio,"[""Yes"", ""No""]",N,2025-01-23 06:57:47


In [6]:
len(df[df["question"] == "question"])

0

In [7]:
my_list = ["question", "pre_ans", "input_type"]
my_var = "pre_ans"

index = my_list.index(my_var) if my_var in my_list else "Not found"
print(index)

1


In [8]:
df_distinct = df[["question", "pre_ans", "input_type"]].drop_duplicates().reset_index(drop=True)
df_distinct

,question,pre_ans,input_type
0,Email address,mboro497@gmail.com,select
1,Phone country code,India (+91),select
2,Mobile phone number,9101925089,text
3,How many years of work experience do you have ...,1,text
4,How many years of work experience do you have ...,2,text
...,...,...,...
171,ECTC,,text
172,LWD,,text
173,Any Speech Processing Experience? Like Speech ...,Select an option,select
174,Hands-on Experience in using speech processing...,Select an option,select


In [9]:
df_distinct["input_type"].unique()

array(['select', 'text', 'radio', 'unknown', 'checkbox'], dtype=object)

In [12]:
df_distinct[df_distinct["input_type"]== 'radio']

,question,pre_ans,input_type
5,Are you comfortable working in an onsite setting?,Yes,radio
7,Are you comfortable commuting to this job's lo...,,radio
14,"Are you willing to undergo a background check,...",Yes,radio
22,Have you completed the following level of educ...,Yes,radio
28,Can you solve LeetCode (medium/hard) coding pr...,,radio
30,Do you have experience in System Design - LLD ...,,radio
31,Do you have experience building distributed sy...,,radio
36,Do you hold a valid work permit for the countr...,,radio
37,Have you been previously employed with Freshwo...,,radio
112,Are you comfortable working in a remote setting?,Yes,radio


# Classification Model

- For select and radio inputs.



# Text generation model:

- For text and unknown inputs.
- transformer-based model like BERT or T5

# Constraints Handling
- Add custom layers or logic to adapt to different input types:
    - For 'select': Use a softmax classifier.
    - For 'text': Use a sequence-to-sequence generator.
    - For 'radio': Treat it like multi-class classification.
    - For 'unknown': Fine-tune for long-form text generation.

https://www.kaggle.com/datasets/stanfordu/stanford-question-answering-dataset?resource=download

In [11]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("stanfordu/stanford-question-answering-dataset")

print("Path to dataset files:", path)

/home/manab/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/manab/.cache/kagglehub/datasets/stanfordu/stanford-question-answering-dataset/versions/2


# Model Use

# form filling work through:
    1. will have a pre loaded pands with q/a if any mathched fill directly
    2. if not check use model to ans